# Escada Tecnológica e Impacto nos Indicadores de Alunos e Professores (Censo Escolar 2023–2025)

**Fase 1 — Exploração Visual e Estatística Completa (Preparação para Modelos de ML em GPU)**

Este notebook investiga, em escala nacional e regional (**Norte, Nordeste, Centro-Oeste, Sudeste e Sul**), como cada degrau de acesso à tecnologia nas escolas brasileiras — desde a **energia elétrica básica** até **conectividade banda larga, notebooks/tablets por aluno e laboratórios** — se correlaciona com:
1. **Índices de Qualificação, Estabilidade e Capacitação Tecnológica dos Professores e Gestores** (`Tabela_Docente_2025_V2`, `Tabela_Gestor_Escolar_2025_v2`);
2. **Índices de Trajetória, Retenção e Distorção Etária dos Alunos** (`Tabela_Matricula_2025_V2`, `Tabela_Turma_2025_V2`);
3. **Evolução Longitudinal das Escolas (2023 → 2024 → 2025)** rastreadas pelo código INEP (`CO_ENTIDADE`).


## 1. Configuração do Ambiente (Google Colab / Local) e Carregamento Otimizado
A célula abaixo detecta automaticamente onde estão as pastas dos microdados (seja rodando localmente ou no **Google Colab** com Google Drive/repositório clonado) e carrega apenas as colunas necessárias (`usecols`) com tipagem otimizada para economizar RAM.


In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")

# Configuração estética global dos gráficos
plt.rcParams["figure.figsize"] = (13, 6)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="viridis")

# Detecção automática dos dados (Local, Compacto .csv.gz ou Google Colab Remoto)
CANDIDATE_DIRS = [
    ".",
    "..",
    "/content",
    "/content/mineracao-educacional",
    "/content/drive/MyDrive/mineracao-educacional",
    "/home/jorge/Documentos/Faculdade/Materias-Faculdade/mineracao-educacional"
]

BASE_DIR = None
COMPACT_2025 = None
COMPACT_LONG = None

for d in CANDIDATE_DIRS:
    if os.path.exists(os.path.join(d, "microdados_censo_escolar_2025_v2")):
        BASE_DIR = os.path.abspath(d)
        break
    for sub in [d, os.path.join(d, "analise_tecnologia")]:
        p25 = os.path.join(sub, "censo_2025_consolidado.csv.gz")
        if os.path.exists(p25):
            COMPACT_2025 = os.path.abspath(p25)
            plong = os.path.join(sub, "censo_2023_2024_longitudinal.csv.gz")
            if os.path.exists(plong):
                COMPACT_LONG = os.path.abspath(plong)
            break

if BASE_DIR is None and COMPACT_2025 is None:
    import urllib.request
    GH_BASE = "https://github.com/JorgRibeiro/mineracao-educacional/raw/main/analise_tecnologia"
    print("Baixando bases compactadas diretamente do GitHub para o servidor do Colab (~2 segundos)...")
    try:
        urllib.request.urlretrieve(f"{GH_BASE}/censo_2025_consolidado.csv.gz", "censo_2025_consolidado.csv.gz")
        urllib.request.urlretrieve(f"{GH_BASE}/censo_2023_2024_longitudinal.csv.gz", "censo_2023_2024_longitudinal.csv.gz")
        COMPACT_2025 = os.path.abspath("censo_2025_consolidado.csv.gz")
        COMPACT_LONG = os.path.abspath("censo_2023_2024_longitudinal.csv.gz")
        print("Download concluído com sucesso!")
    except Exception as e:
        raise RuntimeError(
            f"Não foi possível baixar do GitHub ({e}). Faça 'git push' da pasta analise_tecnologia/ para o repositório!"
        )

if BASE_DIR is not None:
    print(f"Diretório raiz completo detectado: {BASE_DIR}")
    PATH_2023 = os.path.join(BASE_DIR, "microdados_censo_escolar_2023", "dados", "microdados_ed_basica_2023.csv")
    PATH_2024 = os.path.join(BASE_DIR, "microdados_censo_escolar_2024_defeso", "dados", "microdados_ed_basica_2024.csv")
    DIR_2025  = os.path.join(BASE_DIR, "microdados_censo_escolar_2025_v2", "dados")
elif COMPACT_2025 is not None:
    print(f"Modo Compacto Colab ativado! Arquivo 2025: {COMPACT_2025}")


NameError: name 'ignore' is not defined

In [ ]:
# Seleção de colunas da Tabela de Escola (comuns a 2023, 2024 e 2025)
COLS_ESCOLA = [
    # Identificação e Território
    "NU_ANO_CENSO", "CO_ENTIDADE", "NO_ENTIDADE", "NO_REGIAO", "CO_REGIAO",
    "SG_UF", "NO_MUNICIPIO", "CO_MUNICIPIO",
    "TP_SITUACAO_FUNCIONAMENTO", "TP_DEPENDENCIA", "TP_LOCALIZACAO", "TP_LOCALIZACAO_DIFERENCIADA",
    # Degrau 1: Energia Elétrica e Básico
    "IN_ENERGIA_REDE_PUBLICA", "IN_ENERGIA_GERADOR_FOSSIL", "IN_ENERGIA_RENOVAVEL", "IN_ENERGIA_INEXISTENTE",
    "IN_AGUA_POTAVEL", "IN_ESGOTO_INEXISTENTE",
    # Degrau 2: Tecnologia Administrativa e Equipamentos Básicos
    "IN_COMPUTADOR", "IN_EQUIP_IMPRESSORA", "IN_INTERNET", "IN_INTERNET_ADMINISTRATIVO",
    # Degrau 3: Conectividade e Tecnologia Pedagógica (Sala de Aula)
    "IN_BANDA_LARGA", "TP_REDE_LOCAL", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_COMUNIDADE",
    "IN_EQUIP_MULTIMIDIA", "QT_EQUIP_MULTIMIDIA", "IN_EQUIP_LOUSA_DIGITAL", "QT_EQUIP_LOUSA_DIGITAL",
    "IN_EQUIP_TV", "QT_EQUIP_TV", "IN_BIBLIOTECA_SALA_LEITURA", "IN_LABORATORIO_CIENCIAS",
    # Degrau 4: Imersão Digital do Aluno (Laboratórios, Internet p/ Alunos e Dispositivos)
    "IN_LABORATORIO_INFORMATICA", "IN_LABORATORIO_EDUC_PROF",
    "IN_INTERNET_ALUNOS", "IN_ACESSO_INTERNET_COMPUTADOR", "IN_ACES_INTERNET_DISP_PESSOAIS",
    "IN_DESKTOP_ALUNO", "QT_DESKTOP_ALUNO",
    "IN_COMP_PORTATIL_ALUNO", "QT_COMP_PORTATIL_ALUNO",
    "IN_TABLET_ALUNO", "QT_TABLET_ALUNO"
]

COLS_MATRICULA_2025 = [
    "CO_ENTIDADE", "QT_MAT_BAS", "QT_MAT_INF", "QT_MAT_FUND", "QT_MAT_FUND_AI", "QT_MAT_FUND_AF",
    "QT_MAT_FUND_AF_6", "QT_MAT_FUND_AF_9",
    "QT_MAT_MED", "QT_MAT_MED_1", "QT_MAT_MED_2", "QT_MAT_MED_3",
    "QT_MAT_MED_IFTP_CT", "QT_MAT_PROF", "QT_MAT_PROF_TEC", "QT_MAT_EJA", "QT_MAT_ESP",
    "QT_MAT_BAS_0_3", "QT_MAT_BAS_4_5", "QT_MAT_BAS_6_10", "QT_MAT_BAS_11_14",
    "QT_MAT_BAS_15_17", "QT_MAT_BAS_18_MAIS",
    "QT_MAT_BAS_INT", "QT_MAT_FUND_INT", "QT_MAT_MED_INT",
    "QT_MAT_BAS_FEM", "QT_MAT_BAS_MASC", "QT_MAT_BAS_PRETA", "QT_MAT_BAS_PARDA", "QT_MAT_BAS_INDIGENA",
    "QT_TRANSP_PUBLICO"
]

COLS_DOCENTE_2025 = [
    "CO_ENTIDADE", "QT_DOC_BAS",
    "QT_DOC_BAS_ESCO_SUP_GRAD", "QT_DOC_BAS_ESCO_SUP_GRAD_LICEN",
    "QT_DOC_BAS_ESCO_SUP_POS_ESPEC", "QT_DOC_BAS_ESCO_SUP_POS_MESTRA", "QT_DOC_BAS_ESCO_SUP_POS_DOUTO",
    "QT_DOC_BAS_ESCO_SUP_POS_NENHUM",
    "QT_DOC_BAS_ESPEC_EDUC_TIC", "QT_DOC_BAS_ESPEC_NENHUM",
    "QT_DOC_BAS_DISC_INFO_COMPUTACAO",
    "QT_DOC_BAS_VINCULO_CONCUR", "QT_DOC_BAS_VINCULO_CONTRA"
]

COLS_TURMA_2025 = [
    "CO_ENTIDADE", "QT_TUR_BAS", "QT_TUR_FUND_AF", "QT_TUR_FUND_AF_CORRFLUXO",
    "QT_TUR_BAS_INT", "QT_TUR_BAS_DISC_INFO_COMPUTACAO", "QT_TUR_BAS_DISC_PROJETO_DE_VIDA"
]

COLS_GESTOR_2025 = [
    "CO_ENTIDADE", "QT_GEST_BAS", "QT_GEST_BAS_ESCO_SUP_GRAD",
    "QT_GEST_BAS_ESPEC_GESTAO", "QT_GEST_BAS_ESPEC_EDUC_TIC",
    "QT_GEST_BAS_ACESSO_CARGO_ELEIC", "QT_GEST_BAS_ACESSO_CARGO_CONC", "QT_GEST_BAS_ACESSO_CARGO_INDIC"
]

if COMPACT_2025 is not None and os.path.exists(COMPACT_2025):
    print(f"Carregando base consolidada compacta 2025 ({COMPACT_2025})...")
    df_25 = pd.read_csv(COMPACT_2025, sep=";", encoding="utf-8", low_memory=False)
else:
    print("Carregando bases originais de 2025 v2...")
    df_esc_25 = pd.read_csv(os.path.join(DIR_2025, "Tabela_Escola_2025_V2.csv"), sep=";", encoding="latin1", usecols=COLS_ESCOLA, low_memory=False)
    df_mat_25 = pd.read_csv(os.path.join(DIR_2025, "Tabela_Matricula_2025_V2.csv"), sep=";", encoding="latin1", usecols=COLS_MATRICULA_2025, low_memory=False)
    df_doc_25 = pd.read_csv(os.path.join(DIR_2025, "Tabela_Docente_2025_V2.csv"), sep=";", encoding="latin1", usecols=COLS_DOCENTE_2025, low_memory=False)
    df_tur_25 = pd.read_csv(os.path.join(DIR_2025, "Tabela_Turma_2025_V2.csv"), sep=";", encoding="latin1", usecols=COLS_TURMA_2025, low_memory=False)
    df_ges_25 = pd.read_csv(os.path.join(DIR_2025, "Tabela_Gestor_Escolar_2025_v2.csv"), sep=";", encoding="latin1", usecols=COLS_GESTOR_2025, low_memory=False)

    df_25 = df_esc_25[
        (df_esc_25["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (df_esc_25["NO_REGIAO"].notna()) &
        (df_esc_25["NO_REGIAO"].str.strip() != "")
    ].copy()

    df_25 = (
        df_25
        .merge(df_mat_25, on="CO_ENTIDADE", how="left")
        .merge(df_doc_25, on="CO_ENTIDADE", how="left")
        .merge(df_tur_25, on="CO_ENTIDADE", how="left")
        .merge(df_ges_25, on="CO_ENTIDADE", how="left")
    )
    del df_esc_25, df_mat_25, df_doc_25, df_tur_25, df_ges_25
    gc.collect()

print(f"Base consolidada 2025 (Escolas Ativas): {df_25.shape[0]:,} escolas x {df_25.shape[1]} variáveis.")


---
## 2. Construção da "Escada Tecnológica" e Índices de Alunos e Professores

Para evitar distorções por *outliers* (como o código `88888` que o Inep utiliza para censurar valores extremos em contagens de equipamentos/profissionais), limpamos os quantitativos e estruturamos duas métricas complementares de acesso tecnológico:

1. **Degrau da Escada Tecnológica (`DEGRAU_TEC`, de 0 a 4)**:
   - **Degrau 0 — Exclusão Elétrica**: Escola sem energia elétrica (`IN_ENERGIA_INEXISTENTE == 1`).
   - **Degrau 1 — Energia Básica (Sem Digitalização)**: Possui energia elétrica, mas não tem nenhum computador nem acesso à internet.
   - **Degrau 2 — Digitalização Administrativa (Só Secretaria)**: Possui computador e/ou internet, porém **restritos ao uso administrativo**, sem internet pedagógica, sem laboratório e sem computadores/tablets para alunos.
   - **Degrau 3 — Conectividade Pedagógica (Sala de Aula)**: Possui internet para ensino-aprendizagem (`IN_INTERNET_APRENDIZAGEM == 1`) ou banda larga com equipamentos multimídia, mas ainda sem laboratório de informática ativo ou sem dispositivos liberados diretamente para os estudantes.
   - **Degrau 4 — Imersão Digital do Estudante**: Escola com Laboratório de Informática (`IN_LABORATORIO_INFORMATICA == 1`) **ou** combinação de internet para alunos (`IN_INTERNET_ALUNOS == 1`) + dispositivos dedicados aos alunos (Desktops, Notebooks ou Tablets).

2. **Índice Contínuo de Infraestrutura Digital (`SCORE_TEC_0_100`)**:
   - Soma ponderada de 0 a 100 pontos combinando: Energia da Rede Pública (10 pts), Computador (10 pts), Internet (10 pts), Banda Larga (10 pts), Rede Local Wi-Fi/Cabo (10 pts), Internet no Ensino-Aprendizagem (15 pts), Projetor/Lousa Digital (10 pts), Internet para Alunos (10 pts), Laboratório de Informática (10 pts) e Dispositivos por Aluno (5 pts).


In [ ]:
def preparar_features_tecnologia_e_desempenho(df):
    df = df.copy()
    
    # Mapeamentos legíveis
    mapa_dep = {1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada"}
    mapa_loc = {1: "Urbana", 2: "Rural"}
    df["REDE"] = df["TP_DEPENDENCIA"].map(mapa_dep)
    df["TIPO_REDE"] = np.where(df["TP_DEPENDENCIA"] == 4, "Privada", "Pública (Fed/Est/Mun)")
    df["LOCALIZACAO"] = df["TP_LOCALIZACAO"].map(mapa_loc)
    
    # Tratamento de códigos de censura/outliers (88888 -> NaN/0) em contagens de equipamentos
    equip_qt_cols = [
        "QT_DESKTOP_ALUNO", "QT_COMP_PORTATIL_ALUNO", "QT_TABLET_ALUNO",
        "QT_EQUIP_LOUSA_DIGITAL", "QT_EQUIP_MULTIMIDIA", "QT_EQUIP_TV", "QT_COMP_ADMINISTRATIVO"
    ]
    for col in equip_qt_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            df.loc[df[col] >= 88888, col] = 0
            
    # Total de dispositivos para uso dos alunos (Desktops + Notebooks + Tablets)
    df["QT_DISP_ALUNO_TOTAL"] = (
        df["QT_DESKTOP_ALUNO"] + df["QT_COMP_PORTATIL_ALUNO"] + df["QT_TABLET_ALUNO"]
    )
    df["TEM_DISP_ALUNO"] = (df["QT_DISP_ALUNO_TOTAL"] > 0).astype(int)
    df["TEM_NOTEBOOK_OU_TABLET"] = ((df["QT_COMP_PORTATIL_ALUNO"] + df["QT_TABLET_ALUNO"]) > 0).astype(int)
    df["TEM_REDE_LOCAL"] = df["TP_REDE_LOCAL"].isin([1, 2, 3]).astype(int)
    df["TEM_MULTIMIDIA_OU_LOUSA"] = ((df["IN_EQUIP_MULTIMIDIA"] == 1) | (df["IN_EQUIP_LOUSA_DIGITAL"] == 1)).astype(int)
    
    # Classificação nos 5 Degraus da Escada Tecnológica
    en_inex = df["IN_ENERGIA_INEXISTENTE"].fillna(0) == 1
    sem_comp_net = (df["IN_INTERNET"].fillna(0) == 0) & (df["IN_COMPUTADOR"].fillna(0) == 0)
    so_adm = (
        (df["IN_INTERNET_ALUNOS"].fillna(0) == 0) &
        (df["IN_INTERNET_APRENDIZAGEM"].fillna(0) == 0) &
        (df["IN_LABORATORIO_INFORMATICA"].fillna(0) == 0) &
        (df["QT_DISP_ALUNO_TOTAL"] == 0)
    )
    imersao_aluno = (
        (df["IN_LABORATORIO_INFORMATICA"].fillna(0) == 1) |
        ((df["IN_INTERNET_ALUNOS"].fillna(0) == 1) & (df["QT_DISP_ALUNO_TOTAL"] > 0))
    )
    
    condicoes = [
        en_inex,
        ~en_inex & sem_comp_net,
        ~en_inex & ~sem_comp_net & so_adm,
        ~en_inex & ~sem_comp_net & ~so_adm & ~imersao_aluno,
        ~en_inex & ~sem_comp_net & ~so_adm & imersao_aluno
    ]
    df["DEGRAU_NUM"] = np.select(condicoes, [0, 1, 2, 3, 4], default=2)
    
    nomes_degraus = {
        0: "D0: Sem Energia",
        1: "D1: Só Energia (Off-line)",
        2: "D2: Digital Só Secretaria",
        3: "D3: Conectividade Pedagógica",
        4: "D4: Imersão Digital (Aluno)"
    }
    df["DEGRAU_NOME"] = df["DEGRAU_NUM"].map(nomes_degraus)
    
    # Score Contínuo de Infraestrutura Digital (0 a 100)
    df["SCORE_TEC_0_100"] = (
        (df["IN_ENERGIA_INEXISTENTE"].fillna(0) == 0).astype(int) * 10 +
        df["IN_COMPUTADOR"].fillna(0).clip(0, 1) * 10 +
        df["IN_INTERNET"].fillna(0).clip(0, 1) * 10 +
        df["IN_BANDA_LARGA"].fillna(0).clip(0, 1) * 10 +
        df["TEM_REDE_LOCAL"] * 10 +
        df["IN_INTERNET_APRENDIZAGEM"].fillna(0).clip(0, 1) * 15 +
        df["TEM_MULTIMIDIA_OU_LOUSA"] * 10 +
        df["IN_INTERNET_ALUNOS"].fillna(0).clip(0, 1) * 10 +
        df["IN_LABORATORIO_INFORMATICA"].fillna(0).clip(0, 1) * 10 +
        df["TEM_DISP_ALUNO"] * 5
    )
    
    # Métricas de Densidade Tecnológica por Aluno (apenas para escolas com matrículas > 0)
    mat_valida = df["QT_MAT_BAS"].replace(0, np.nan)
    df["DISP_POR_100_ALUNOS"] = (df["QT_DISP_ALUNO_TOTAL"] / mat_valida * 100).clip(upper=100)
    df["ALUNOS_POR_DISP"] = np.where(df["QT_DISP_ALUNO_TOTAL"] > 0, mat_valida / df["QT_DISP_ALUNO_TOTAL"], np.nan)
    
    # --- INDICADORES DE DESEMPENHO / QUALIFICAÇÃO DOCENTE (2025) ---
    if "QT_DOC_BAS" in df.columns:
        doc_valido = df["QT_DOC_BAS"].replace(0, np.nan)
        df["PCT_DOC_SUPERIOR"] = (df["QT_DOC_BAS_ESCO_SUP_GRAD"] / doc_valido * 100).clip(0, 100)
        df["PCT_DOC_LICENCIATURA"] = (df["QT_DOC_BAS_ESCO_SUP_GRAD_LICEN"] / doc_valido * 100).clip(0, 100)
        df["PCT_DOC_POS_GRAD"] = (
            (df["QT_DOC_BAS_ESCO_SUP_POS_ESPEC"].fillna(0) +
             df["QT_DOC_BAS_ESCO_SUP_POS_MESTRA"].fillna(0) +
             df["QT_DOC_BAS_ESCO_SUP_POS_DOUTO"].fillna(0)) / doc_valido * 100
        ).clip(0, 100)
        df["PCT_DOC_MESTRADO_DOUTORADO"] = (
            (df["QT_DOC_BAS_ESCO_SUP_POS_MESTRA"].fillna(0) +
             df["QT_DOC_BAS_ESCO_SUP_POS_DOUTO"].fillna(0)) / doc_valido * 100
        ).clip(0, 100)
        df["PCT_DOC_FORMACAO_TIC"] = (df["QT_DOC_BAS_ESPEC_EDUC_TIC"] / doc_valido * 100).clip(0, 100)
        df["TEM_DOC_FORMACAO_TIC"] = (df["QT_DOC_BAS_ESPEC_EDUC_TIC"].fillna(0) > 0).astype(int)
        df["TEM_DOC_COMPUTACAO"] = (df["QT_DOC_BAS_DISC_INFO_COMPUTACAO"].fillna(0) > 0).astype(int)
        df["PCT_DOC_CONCURSADO"] = (df["QT_DOC_BAS_VINCULO_CONCUR"] / doc_valido * 100).clip(0, 100)
        
    # --- INDICADORES DE DESEMPENHO / TRAJETÓRIA DOS ESTUDANTES ---
    if "QT_MAT_BAS" in df.columns:
        df["PCT_MAT_TEMPO_INTEGRAL"] = (df["QT_MAT_BAS_INT"] / mat_valida * 100).clip(0, 100)
        
        # 1. Proxy de Distorção Idade-Série (Atraso Escolar no Ensino Fundamental):
        # Em escolas exclusivamente de Ed. Infantil/Fundamental (sem EM, EJA ou Profissional),
        # qualquer aluno com 15+ anos está em atraso escolar (distorção idade-série).
        mask_so_ef = (
            (df["QT_MAT_FUND"].fillna(0) >= 15) &
            (df["QT_MAT_MED"].fillna(0) == 0) &
            (df["QT_MAT_EJA"].fillna(0) == 0) &
            (df["QT_MAT_PROF"].fillna(0) == 0)
        )
        atraso_ef = (df["QT_MAT_BAS_15_17"].fillna(0) + df["QT_MAT_BAS_18_MAIS"].fillna(0)) / df["QT_MAT_FUND"] * 100
        df["TAXA_ATRASO_ETARIO_EF"] = np.where(mask_so_ef, atraso_ef.clip(0, 100), np.nan)
        
        # 2. Taxa de Retenção/Sobrevivência no Ensino Médio (Matrículas na 3ª série / 1ª série)
        mask_em = (df["QT_MAT_MED_1"].fillna(0) >= 10) & (df["QT_MAT_MED_3"].fillna(0) > 0)
        ret_em = (df["QT_MAT_MED_3"] / df["QT_MAT_MED_1"]) * 100
        df["TAXA_SOBREVIVENCIA_EM_3_1"] = np.where(mask_em, ret_em.clip(0, 130), np.nan)
        
        # 3. Taxa de Conclusão dos Anos Finais do Fundamental (9º ano / 6º ano)
        mask_af = (df["QT_MAT_FUND_AF_6"].fillna(0) >= 10) & (df["QT_MAT_FUND_AF_9"].fillna(0) > 0)
        ret_af = (df["QT_MAT_FUND_AF_9"] / df["QT_MAT_FUND_AF_6"]) * 100
        df["TAXA_SOBREVIVENCIA_AF_9_6"] = np.where(mask_af, ret_af.clip(0, 130), np.nan)
        
        # 4. Presença de Turmas de Correção de Fluxo (defasagem escolar reconhecida)
        if "QT_TUR_FUND_AF_CORRFLUXO" in df.columns:
            df["TEM_TURMA_CORRECAO_FLUXO"] = np.where(
                df["QT_TUR_FUND_AF"].fillna(0) > 0,
                (df["QT_TUR_FUND_AF_CORRFLUXO"].fillna(0) > 0).astype(int) * 100,
                np.nan
            )
            
    return df

df_25 = preparar_features_tecnologia_e_desempenho(df_25)
print("Resumo da distribuição de escolas por Degrau Tecnológico (Brasil 2025):")
resumo_deg = df_25["DEGRAU_NOME"].value_counts().sort_index()
for deg, qtd in resumo_deg.items():
    print(f"  {deg:<32}: {qtd:>7,} escolas ({qtd/len(df_25)*100:5.1f}%)")


---
## 3. Radiografia Regional da Escada Tecnológica (As 5 Regiões do Brasil)

Aqui visualizamos **onde estão os gargalos da escada tecnológica** em cada macrorregião (**Norte, Nordeste, Centro-Oeste, Sudeste e Sul**):
- **Gráfico 1**: Proporção das escolas em cada um dos 5 Degraus Tecnológicos por Região e por Localização (Urbana vs. Rural).
- **Gráfico 2 (A Escada Completa item a item)**: Um *heatmap* mostrando o percentual exato de escolas que possuem cada tecnologia individual (Energia → Computador Administrativo → Internet → Banda Larga → Rede Wi-Fi/Cabo → Projetor/Lousa → Internet no Ensino → Internet p/ Alunos → Notebooks/Tablets → Laboratório de Informática) em cada região.


In [ ]:
# GRÁFICO 1: Distribuição dos Degraus Tecnológicos por Região e Localização
ORDEM_REGIOES = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
ORDEM_DEGRAUS = [
    "D0: Sem Energia",
    "D1: Só Energia (Off-line)",
    "D2: Digital Só Secretaria",
    "D3: Conectividade Pedagógica",
    "D4: Imersão Digital (Aluno)"
]
CORES_DEGRAUS = ["#d73027", "#fc8d59", "#fee08b", "#91bfdb", "#1a9850"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), sharey=False)

# Painel A: Todas as escolas ativas por Região
ct_reg = pd.crosstab(df_25["NO_REGIAO"], df_25["DEGRAU_NOME"], normalize="index").reindex(ORDEM_REGIOES)[ORDEM_DEGRAUS] * 100
ct_reg.plot(kind="barh", stacked=True, color=CORES_DEGRAUS, ax=axes[0], width=0.72, edgecolor="white")
axes[0].set_title("A) Distribuição da Escada Tecnológica por Região (Todas as Escolas)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("% das Escolas em Atividade (2025)")
axes[0].set_ylabel("Região Geográfica")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend().set_visible(False)

for n, (reg, row) in enumerate(ct_reg.iterrows()):
    acum = 0
    for val in row:
        if val >= 5.5:
            axes[0].text(acum + val/2, n, f"{val:.1f}%", ha="center", va="center", color="black", fontweight="bold", fontsize=9.5)
        acum += val

# Painel B: Apenas Escolas PÚBLICAS RURAIS por Região (onde a exclusão é mais crítica)
df_rural_pub = df_25[(df_25["TP_DEPENDENCIA"] != 4) & (df_25["TP_LOCALIZACAO"] == 2)]
ct_rur = pd.crosstab(df_rural_pub["NO_REGIAO"], df_rural_pub["DEGRAU_NOME"], normalize="index").reindex(ORDEM_REGIOES)[ORDEM_DEGRAUS] * 100
ct_rur.plot(kind="barh", stacked=True, color=CORES_DEGRAUS, ax=axes[1], width=0.72, edgecolor="white")
axes[1].set_title("B) Foco nas Escolas Públicas Rurais por Região (Desigualdade Extrema)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("% das Escolas Públicas Rurais (2025)")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(title="Degrau da Escada Tecnológica", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

for n, (reg, row) in enumerate(ct_rur.iterrows()):
    acum = 0
    for val in row:
        if val >= 5.5:
            axes[1].text(acum + val/2, n, f"{val:.1f}%", ha="center", va="center", color="black", fontweight="bold", fontsize=9.5)
        acum += val

plt.suptitle("Desigualdade Tecnológica Regional na Educação Básica Brasileira (Censo 2025)", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# GRÁFICO 2: Heatmap "Degrau por Degrau" - Cobertura de Cada Tecnologia por Região
metricas_escada = {
    "1. Energia Elétrica (Qualquer fonte)": (df_25["IN_ENERGIA_INEXISTENTE"] == 0),
    "2. Energia da Rede Pública": (df_25["IN_ENERGIA_REDE_PUBLICA"] == 1),
    "3. Computador na Escola (Adm/Geral)": (df_25["IN_COMPUTADOR"] == 1),
    "4. Acesso à Internet (Qualquer uso)": (df_25["IN_INTERNET"] == 1),
    "5. Internet Banda Larga": (df_25["IN_BANDA_LARGA"] == 1),
    "6. Rede Local (Wi-Fi / Cabo)": (df_25["TEM_REDE_LOCAL"] == 1),
    "7. Projetor Multimídia ou Lousa Digital": (df_25["TEM_MULTIMIDIA_OU_LOUSA"] == 1),
    "8. Internet p/ Ensino e Aprendizagem": (df_25["IN_INTERNET_APRENDIZAGEM"] == 1),
    "9. Computadores/Notebooks/Tablets p/ Alunos": (df_25["TEM_DISP_ALUNO"] == 1),
    "10. Notebooks ou Tablets p/ Alunos": (df_25["TEM_NOTEBOOK_OU_TABLET"] == 1),
    "11. Internet Liberada para Alunos": (df_25["IN_INTERNET_ALUNOS"] == 1),
    "12. Laboratório de Informática": (df_25["IN_LABORATORIO_INFORMATICA"] == 1),
    "13. Laboratório de Ciências": (df_25["IN_LABORATORIO_CIENCIAS"] == 1),
}

tabela_escada = pd.DataFrame({nome: cond.groupby(df_25["NO_REGIAO"]).mean() * 100 for nome, cond in metricas_escada.items()}).reindex(ORDEM_REGIOES).T
tabela_escada["BRASIL (Média)"] = pd.Series({nome: cond.mean() * 100 for nome, cond in metricas_escada.items()})

plt.figure(figsize=(13, 7.5))
sns.heatmap(
    tabela_escada,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    cbar_kws={"label": "% das Escolas em Atividade"},
    linewidths=0.6
)
plt.title("A Escada Completa de Infraestrutura Tecnológica por Região (% de Escolas com Acesso - 2025)", fontsize=14, fontweight="bold", pad=14)
plt.xlabel("Região Geográfica")
plt.ylabel("Item da Escada Tecnológica (do Básico ao Avançado)")
plt.tight_layout()
plt.show()


### 3.1. A "Ilusão do Equipamento": Quantos Alunos Disputam Cada Computador/Notebook?
Muitas escolas marcam "Sim" para computadores ou laboratório de informática, mas possuem apenas 2 ou 3 máquinas para centenas de estudantes. O gráfico abaixo analisa a **densidade real de dispositivos (`Desktops + Notebooks + Tablets`) por 100 alunos** nas escolas públicas e privadas de cada região.


In [ ]:
# GRÁFICO 3: Dispositivos por 100 Alunos e Composição (Desktop vs Notebook vs Tablet) por Região
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel A: Dispositivos por 100 alunos (Pública vs Privada por Região)
sns.barplot(
    data=df_25[df_25["QT_MAT_BAS"] > 0],
    x="NO_REGIAO",
    y="DISP_POR_100_ALUNOS",
    hue="TIPO_REDE",
    order=ORDEM_REGIOES,
    palette=["#2b83ba", "#fdae61"],
    ax=axes[0],
    errorbar="ci"
)
axes[0].set_title("A) Densidade: Dispositivos de Uso do Aluno p/ cada 100 Matrículas", fontsize=12.5, fontweight="bold")
axes[0].set_xlabel("Região Geográfica")
axes[0].set_ylabel("Média de Dispositivos por 100 Alunos")
axes[0].legend(title="Tipo de Rede")

# Painel B: Mix de Tecnologia para Alunos (Desktop vs Notebook vs Tablet por 1.000 alunos)
df_valid_mat = df_25[df_25["QT_MAT_BAS"] > 0].groupby("NO_REGIAO")[
    ["QT_DESKTOP_ALUNO", "QT_COMP_PORTATIL_ALUNO", "QT_TABLET_ALUNO", "QT_MAT_BAS"]
].sum().reindex(ORDEM_REGIOES)

mix_1000 = pd.DataFrame({
    "Desktops de Mesa": df_valid_mat["QT_DESKTOP_ALUNO"] / df_valid_mat["QT_MAT_BAS"] * 1000,
    "Notebooks / Portáteis": df_valid_mat["QT_COMP_PORTATIL_ALUNO"] / df_valid_mat["QT_MAT_BAS"] * 1000,
    "Tablets": df_valid_mat["QT_TABLET_ALUNO"] / df_valid_mat["QT_MAT_BAS"] * 1000,
}, index=ORDEM_REGIOES)

mix_1000.plot(kind="bar", stacked=True, ax=axes[1], color=["#3288bd", "#66c2a5", "#abdda4"], width=0.65, edgecolor="black")
axes[1].set_title("B) Composição do Parque Tecnológico Estudantil (por 1.000 Alunos)", fontsize=12.5, fontweight="bold")
axes[1].set_xlabel("Região Geográfica")
axes[1].set_ylabel("Equipamentos para cada 1.000 Alunos Matriculados")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Tipo de Dispositivo")

plt.suptitle("Densidade e Composição dos Dispositivos Digitais para Estudantes (2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 4. Impacto da Escada Tecnológica nos Índices dos Professores e Gestores

Como o Censo Escolar 2025 v2 traz as tabelas inéditas `Tabela_Docente_2025_V2` e `Tabela_Gestor_Escolar_2025_v2`, podemos responder:
1. **À medida que a escola sobe do Degrau 0 (Sem Energia) ao Degrau 4 (Imersão Digital), como mudam os indicadores de qualificação dos professores?**
   - Percentual de professores com **Ensino Superior Completo** (`PCT_DOC_SUPERIOR`);
   - Percentual de professores com **Pós-Graduação — Especialização, Mestrado ou Doutorado** (`PCT_DOC_POS_GRAD`);
   - Percentual de professores **Concursados/Efetivos** na rede pública (`PCT_DOC_CONCURSADO`);
2. **O Paradoxo "Hardware sem Treinamento"**:
   - Quantas escolas possuem tecnologia de ponta (Degrau 4), mas não têm professores ou gestores com formação continuada em TICs (`QT_DOC_BAS_ESPEC_EDUC_TIC`, `QT_GEST_BAS_ESPEC_EDUC_TIC`)?


In [ ]:
# GRÁFICO 4: Qualificação Docente ao Subir a Escada Tecnológica (Brasil e por Região)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel A: Evolução dos 3 grandes indicadores docentes por Degrau Tecnológico
doc_por_degrau = df_25.groupby("DEGRAU_NOME")[
    ["PCT_DOC_SUPERIOR", "PCT_DOC_POS_GRAD", "PCT_DOC_CONCURSADO"]
].mean().reindex(ORDEM_DEGRAUS)

x = np.arange(len(ORDEM_DEGRAUS))
labels_curtos = ["D0: Sem\nEnergia", "D1: Só\nEnergia", "D2: Só\nSecretaria", "D3: Conect.\nPedagógica", "D4: Imersão\nAluno"]

axes[0].plot(x, doc_por_degrau["PCT_DOC_SUPERIOR"], marker="o", linewidth=2.8, markersize=8, label="% Docentes c/ Ensino Superior", color="#1b9e77")
axes[0].plot(x, doc_por_degrau["PCT_DOC_POS_GRAD"], marker="s", linewidth=2.8, markersize=8, label="% Docentes c/ Pós-Graduação (Esp/Mest/Dout)", color="#d95f02")
axes[0].plot(x, doc_por_degrau["PCT_DOC_CONCURSADO"], marker="^", linewidth=2.8, markersize=8, label="% Docentes Concursados/Efetivos", color="#7570b3")

for idx_x in x:
    v_sup = doc_por_degrau["PCT_DOC_SUPERIOR"].iloc[idx_x]
    v_pos = doc_por_degrau["PCT_DOC_POS_GRAD"].iloc[idx_x]
    axes[0].text(idx_x, v_sup + 1.8, f"{v_sup:.1f}%", ha="center", fontsize=9.5, fontweight="bold", color="#1b9e77")
    axes[0].text(idx_x, v_pos + 1.8, f"{v_pos:.1f}%", ha="center", fontsize=9.5, fontweight="bold", color="#d95f02")

axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_curtos)
axes[0].set_ylim(0, 105)
axes[0].set_title("A) Salto na Qualificação Docente por Degrau Tecnológico", fontsize=12.5, fontweight="bold")
axes[0].set_ylabel("Média por Escola (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend(loc="lower right")

# Painel B: % de Professores com Pós-Graduação por Degrau e Região (Apenas Escolas Públicas!)
df_pub = df_25[df_25["TP_DEPENDENCIA"] != 4]
sns.lineplot(
    data=df_pub,
    x="DEGRAU_NUM",
    y="PCT_DOC_POS_GRAD",
    hue="NO_REGIAO",
    hue_order=ORDEM_REGIOES,
    style="NO_REGIAO",
    markers=True,
    dashes=False,
    linewidth=2.5,
     markersize=8,
    ax=axes[1]
)
axes[1].set_xticks([0, 1, 2, 3, 4])
axes[1].set_xticklabels(labels_curtos)
axes[1].set_title("B) Docentes Pós-Graduados na Rede Pública por Região e Degrau", fontsize=12.5, fontweight="bold")
axes[1].set_xlabel("Degrau da Escada Tecnológica")
axes[1].set_ylabel("% de Docentes com Pós-Graduação")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(title="Região (Rede Pública)")

plt.suptitle("Correlação entre Infraestrutura Tecnológica e Qualificação dos Professores (2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# GRÁFICO 5: Capacitação Docente em TICs e Aulas de Computação vs Escada Tecnológica
fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))

# Painel A: % de Escolas com pelo menos 1 Professor Formado em TICs e 1 Professor de Computação
tic_deg = df_25.groupby("DEGRAU_NOME")[["TEM_DOC_FORMACAO_TIC", "TEM_DOC_COMPUTACAO"]].mean().reindex(ORDEM_DEGRAUS) * 100
tic_deg.columns = ["Escola tem Docente c/ Formação Continuada em TIC (80h+)", "Escola tem Docente Lecionando Informática/Computação"]
tic_deg.plot(kind="bar", ax=axes[0], color=["#2c7bb6", "#fdae61"], width=0.68, edgecolor="black")
axes[0].set_xticklabels(labels_curtos, rotation=0)
axes[0].set_title("A) Presença de Professores Capacitados em TIC / Computação por Degrau", fontsize=12, fontweight="bold")
axes[0].set_ylabel("% das Escolas")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend()

# Painel B: O Paradoxo nas Escolas de Degrau 4 (Imersão Digital) por Região
df_d4 = df_25[df_25["DEGRAU_NUM"] == 4]
tic_reg_d4 = df_d4.groupby("NO_REGIAO")[["TEM_DOC_FORMACAO_TIC", "TEM_DOC_COMPUTACAO"]].mean().reindex(ORDEM_REGIOES) * 100
tic_reg_d4.columns = ["Docente Formado em TIC (80h+)", "Docente de Informática/Computação"]
tic_reg_d4.plot(kind="bar", ax=axes[1], color=["#2c7bb6", "#fdae61"], width=0.68, edgecolor="black")
axes[1].set_xticklabels(ORDEM_REGIOES, rotation=0)
axes[1].set_title("B) Escolas no Topo (Degrau 4): % com Docentes de TIC por Região", fontsize=12, fontweight="bold")
axes[1].set_ylabel("% das Escolas do Degrau 4")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.suptitle("Hardware vs. Capital Humano: Capacitação Tecnológica Docente (2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 5. Impacto da Tecnologia na Vida e no Desempenho dos Estudantes

Mesmo controlando apenas para a **Rede Pública** (para isolar o efeito socioeconômico das escolas privadas), o avanço na Escada Tecnológica mostra correlações fortíssimas com os indicadores estudantis:
1. **Queda drástica do Atraso Escolar (Distorção Idade-Série no Ensino Fundamental)**:
   - No **Degrau 0 (Sem Energia)**, mais de **15%** dos alunos de escolas exclusivas de Ensino Fundamental têm **15 anos ou mais** (idade em que já deveriam estar no Ensino Médio). Nos **Degraus 3 e 4**, essa taxa cai para **~2% a 3%**!
2. **Aumento da Taxa de Retenção/Sobrevivência no Ensino Médio (`Matrículas 3ª Série / 1ª Série`)**:
   - Mede a capacidade da escola de fazer os alunos que entram no 1º ano chegarem ao 3º ano do Ensino Médio sem evadir ou ficar retidos.
3. **Expansão do Ensino em Tempo Integral e da Educação Profissional/Técnica Integrada**.


In [ ]:
# GRÁFICO 6: Impacto nos Alunos (1) - Queda do Atraso Etário no Ensino Fundamental e Aumento da Retenção no Ensino Médio
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel A: Taxa de Atraso Etário no Ensino Fundamental (Alunos 15+ anos no EF) por Degrau e Região
sns.barplot(
    data=df_25[df_25["TAXA_ATRASO_ETARIO_EF"].notna()],
    x="DEGRAU_NUM",
    y="TAXA_ATRASO_ETARIO_EF",
    hue="NO_REGIAO",
    hue_order=ORDEM_REGIOES,
    palette="magma",
    ax=axes[0],
    errorbar=None
)
axes[0].set_xticks([0, 1, 2, 3, 4])
axes[0].set_xticklabels(labels_curtos)
axes[0].set_title("A) Queda do Atraso Escolar no Fundamental conforme Sobe o Degrau", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Degrau da Escada Tecnológica")
axes[0].set_ylabel("% de Alunos com 15+ anos retidos no Fundamental")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend(title="Região")

# Painel B: Taxa de Sobrevivência/Retenção no Ensino Médio (3ª Série / 1ª Série) e Ensino Integral por Degrau
est_por_deg = df_25.groupby("DEGRAU_NUM")[
    ["TAXA_SOBREVIVENCIA_EM_3_1", "TAXA_SOBREVIVENCIA_AF_9_6", "PCT_MAT_TEMPO_INTEGRAL"]
].mean()

axes[1].plot(est_por_deg.index, est_por_deg["TAXA_SOBREVIVENCIA_AF_9_6"], marker="o", linewidth=2.8, markersize=8, label="Sobrevivência Anos Finais EF (9º ano / 6º ano)", color="#2b83ba")
axes[1].plot(est_por_deg.index, est_por_deg["TAXA_SOBREVIVENCIA_EM_3_1"], marker="s", linewidth=2.8, markersize=8, label="Sobrevivência Ensino Médio (3ª série / 1ª série)", color="#1a9850")
axes[1].plot(est_por_deg.index, est_por_deg["PCT_MAT_TEMPO_INTEGRAL"], marker="^", linewidth=2.8, markersize=8, label="% Matrículas em Tempo Integral", color="#d7191c")

for idx_x in est_por_deg.index:
    v_em = est_por_deg.loc[idx_x, "TAXA_SOBREVIVENCIA_EM_3_1"]
    v_int = est_por_deg.loc[idx_x, "PCT_MAT_TEMPO_INTEGRAL"]
    axes[1].text(idx_x, v_em + 1.8, f"{v_em:.1f}%", ha="center", fontweight="bold", color="#1a9850", fontsize=9.5)
    axes[1].text(idx_x, v_int + 1.8, f"{v_int:.1f}%", ha="center", fontweight="bold", color="#d7191c", fontsize=9.5)

axes[1].set_xticks([0, 1, 2, 3, 4])
axes[1].set_xticklabels(labels_curtos)
axes[1].set_ylim(0, 105)
axes[1].set_title("B) Progressão/Sobrevivência de Alunos e Tempo Integral por Degrau", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Degrau da Escada Tecnológica")
axes[1].set_ylabel("Taxa Média por Escola (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(loc="lower right")

plt.suptitle("Impacto da Escada Tecnológica na Trajetória e Permanência dos Estudantes (2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### 5.1. O Impacto Individual de Cada Tecnologia (Decomposição da Escada)
Qual tecnologia individual tem o maior impacto associado à redução do atraso escolar dos alunos e ao aumento da qualificação docente? Abaixo comparamos a diferença média (com vs. sem a tecnologia) **apenas dentro das Escolas Públicas** para evitar viés da rede privada.


In [ ]:
# GRÁFICO 7: Impacto Marginal de Cada Tecnologia nas Escolas Públicas (Com vs. Sem a Tecnologia)
df_pub = df_25[df_25["TP_DEPENDENCIA"] != 4].copy()

tecnologias_binarias = [
    ("Energia Rede Pública", df_pub["IN_ENERGIA_REDE_PUBLICA"] == 1),
    ("Computador na Escola", df_pub["IN_COMPUTADOR"] == 1),
    ("Acesso à Internet", df_pub["IN_INTERNET"] == 1),
    ("Internet Banda Larga", df_pub["IN_BANDA_LARGA"] == 1),
    ("Rede Local (Wi-Fi/Cabo)", df_pub["TEM_REDE_LOCAL"] == 1),
    ("Projetor / Lousa Digital", df_pub["TEM_MULTIMIDIA_OU_LOUSA"] == 1),
    ("Internet no Ensino-Aprendizagem", df_pub["IN_INTERNET_APRENDIZAGEM"] == 1),
    ("Notebooks / Tablets p/ Alunos", df_pub["TEM_NOTEBOOK_OU_TABLET"] == 1),
    ("Internet Liberada p/ Alunos", df_pub["IN_INTERNET_ALUNOS"] == 1),
    ("Laboratório de Informática", df_pub["IN_LABORATORIO_INFORMATICA"] == 1),
    ("Laboratório de Ciências", df_pub["IN_LABORATORIO_CIENCIAS"] == 1),
]

linhas_impacto = []
for nome_tec, mask_sim in tecnologias_binarias:
    atraso_sim = df_pub.loc[mask_sim, "TAXA_ATRASO_ETARIO_EF"].mean()
    atraso_nao = df_pub.loc[~mask_sim, "TAXA_ATRASO_ETARIO_EF"].mean()
    
    pos_sim = df_pub.loc[mask_sim, "PCT_DOC_POS_GRAD"].mean()
    pos_nao = df_pub.loc[~mask_sim, "PCT_DOC_POS_GRAD"].mean()
    
    em_sim = df_pub.loc[mask_sim, "TAXA_SOBREVIVENCIA_EM_3_1"].mean()
    em_nao = df_pub.loc[~mask_sim, "TAXA_SOBREVIVENCIA_EM_3_1"].mean()
    
    linhas_impacto.append({
        "Tecnologia": nome_tec,
        "Redução no Atraso Escolar EF (p.p.)": atraso_nao - atraso_sim,
        "Ganho em Docentes Pós-Graduados (p.p.)": pos_sim - pos_nao,
        "Ganho na Sobrevivência do Ensino Médio (p.p.)": em_sim - em_nao
    })

df_impacto = pd.DataFrame(linhas_impacto).set_index("Tecnologia")

fig, axes = plt.subplots(1, 2, figsize=(16, 6.2), sharey=True)

df_impacto["Redução no Atraso Escolar EF (p.p.)"].plot(kind="barh", ax=axes[0], color="#d95f02", edgecolor="black")
axes[0].set_title("A) Redução do Atraso Escolar no EF (Sem vs. Com a Tecnologia)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Redução no Atraso Etário (pontos percentuais - Rede Pública)")
axes[0].axvline(0, color="black", linewidth=1)

df_impacto["Ganho em Docentes Pós-Graduados (p.p.)"].plot(kind="barh", ax=axes[1], color="#1b9e77", edgecolor="black")
axes[1].set_title("B) Aumento de Professores Pós-Graduados (Com vs. Sem a Tecnologia)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Aumento em Docentes c/ Pós-Graduação (pontos percentuais - Rede Pública)")
axes[1].axvline(0, color="black", linewidth=1)

plt.suptitle("Impacto Associado a Cada Degrau Tecnológico Dentro da Rede Pública (2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 6. Evolução Longitudinal (2023 → 2024 → 2025): O Brasil Subiu a Escada?

Agora carregamos as colunas tecnológicas de **2023** e **2024** para comparar a evolução histórica das escolas brasileiras nos 3 anos disponíveis no repositório e medir **quantas escolas subiram de degrau** em cada região do país.


In [ ]:
# Carregamento leve das colunas tecnológicas de 2023 e 2024 para análise longitudinal
COLS_LONG = [
    "NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "TP_SITUACAO_FUNCIONAMENTO", "TP_DEPENDENCIA", "TP_LOCALIZACAO",
    "IN_ENERGIA_INEXISTENTE", "IN_COMPUTADOR", "IN_INTERNET", "IN_BANDA_LARGA", "TP_REDE_LOCAL",
    "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "IN_LABORATORIO_INFORMATICA",
    "IN_EQUIP_MULTIMIDIA", "IN_EQUIP_LOUSA_DIGITAL",
    "QT_DESKTOP_ALUNO", "QT_COMP_PORTATIL_ALUNO", "QT_TABLET_ALUNO", "QT_MAT_BAS"
]

if COMPACT_LONG is not None and os.path.exists(COMPACT_LONG):
    print(f"Carregando base longitudinal compacta 2023-2024 ({COMPACT_LONG})...")
    df_23_24 = pd.read_csv(COMPACT_LONG, sep=";", encoding="utf-8", low_memory=False)
    df_23_24 = preparar_features_tecnologia_e_desempenho(df_23_24)
    df_painel = pd.concat([
        df_23_24[["NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "DEGRAU_NUM", "SCORE_TEC_0_100", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "TEM_NOTEBOOK_OU_TABLET"]],
        df_25[["NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "DEGRAU_NUM", "SCORE_TEC_0_100", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "TEM_NOTEBOOK_OU_TABLET"]]
    ], ignore_index=True)
    del df_23_24
else:
    df_23_long = pd.read_csv(PATH_2023, sep=";", encoding="latin1", usecols=COLS_LONG, low_memory=False)
    df_24_long = pd.read_csv(PATH_2024, sep=";", encoding="latin1", usecols=COLS_LONG, low_memory=False)

    df_23_long = df_23_long[(df_23_long["TP_SITUACAO_FUNCIONAMENTO"] == 1) & (df_23_long["NO_REGIAO"].notna())].copy()
    df_24_long = df_24_long[(df_24_long["TP_SITUACAO_FUNCIONAMENTO"] == 1) & (df_24_long["NO_REGIAO"].notna())].copy()

    df_23_long = preparar_features_tecnologia_e_desempenho(df_23_long)
    df_24_long = preparar_features_tecnologia_e_desempenho(df_24_long)

    df_painel = pd.concat([
        df_23_long[["NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "DEGRAU_NUM", "SCORE_TEC_0_100", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "TEM_NOTEBOOK_OU_TABLET"]],
        df_24_long[["NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "DEGRAU_NUM", "SCORE_TEC_0_100", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "TEM_NOTEBOOK_OU_TABLET"]],
        df_25[["NU_ANO_CENSO", "CO_ENTIDADE", "NO_REGIAO", "DEGRAU_NUM", "SCORE_TEC_0_100", "IN_INTERNET_APRENDIZAGEM", "IN_INTERNET_ALUNOS", "TEM_NOTEBOOK_OU_TABLET"]]
    ], ignore_index=True)
    del df_23_long, df_24_long
gc.collect()

# GRÁFICO 8: Evolução do Score Tecnológico (2023 -> 2024 -> 2025) por Região e Matriz de Transição
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel A: Evolução do Score Médio de Infraestrutura Digital (0 a 100) por Região (2023 a 2025)
sns.lineplot(
    data=df_painel,
    x="NU_ANO_CENSO",
    y="SCORE_TEC_0_100",
    hue="NO_REGIAO",
    hue_order=ORDEM_REGIOES,
    style="NO_REGIAO",
    markers=True,
    dashes=False,
    linewidth=2.8,
    markersize=9,
    ax=axes[0]
)
axes[0].set_xticks([2023, 2024, 2025])
axes[0].set_title("A) Evolução do Score de Infraestrutura Digital por Região (2023–2025)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Ano do Censo Escolar")
axes[0].set_ylabel("Score Médio de Infraestrutura Digital (0 a 100)")
axes[0].legend(title="Região")

# Painel B: Matriz de Transição das Mesmas Escolas (2023 vs 2025 via CO_ENTIDADE)
p23 = df_painel[df_painel["NU_ANO_CENSO"] == 2023].set_index("CO_ENTIDADE")["DEGRAU_NUM"]
p25 = df_painel[df_painel["NU_ANO_CENSO"] == 2025].set_index("CO_ENTIDADE")["DEGRAU_NUM"]
comuns = p23.index.intersection(p25.index)

matriz_trans = pd.crosstab(p23.loc[comuns], p25.loc[comuns], normalize="index") * 100
matriz_trans.index = [f"2023: D{i}" for i in matriz_trans.index]
matriz_trans.columns = [f"2025: D{i}" for i in matriz_trans.columns]

sns.heatmap(matriz_trans, annot=True, fmt=".1f", cmap="Greens", ax=axes[1], cbar_kws={"label": "% das Escolas de 2023"})
axes[1].set_title(f"B) Matriz de Transição de Degrau (2023 → 2025 | n={len(comuns):,} escolas)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Degrau Alcançado em 2025")
axes[1].set_ylabel("Degrau de Origem em 2023")

plt.suptitle("Dinâmica Longitudinal da Tecnologia Escolar no Brasil (2023–2025)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 7. Matriz de Correlação Multivariada e Preparação para Fase 2 (Machine Learning)

Por fim, consolidamos a matriz de correlação de Spearman entre o **Score Tecnológico (`SCORE_TEC_0_100`)**, a **Densidade de Dispositivos por Aluno**, os **Indicadores Docentes** e os **Indicadores de Trajetória Estudantil**, deixando o DataFrame `df_ml_ready` limpo e pronto para a **Fase 2 (Modelos Preditivos e Explicabilidade SHAP em GPU)**.


In [ ]:
# GRÁFICO 9: Matriz de Correlação (Spearman) entre Escada Tecnológica, Professores e Alunos
cols_corr = {
    "DEGRAU_NUM": "Degrau Escada Tec. (0-4)",
    "SCORE_TEC_0_100": "Score Tec. (0-100)",
    "DISP_POR_100_ALUNOS": "Dispositivos / 100 Alunos",
    "PCT_DOC_SUPERIOR": "% Docentes c/ Superior",
    "PCT_DOC_POS_GRAD": "% Docentes c/ Pós-Grad.",
    "PCT_DOC_CONCURSADO": "% Docentes Concursados",
    "PCT_DOC_FORMACAO_TIC": "% Docentes Formados TIC",
    "PCT_MAT_TEMPO_INTEGRAL": "% Alunos Tempo Integral",
    "TAXA_SOBREVIVENCIA_AF_9_6": "Sobrevivência EF (9º/6º)",
    "TAXA_SOBREVIVENCIA_EM_3_1": "Sobrevivência EM (3ª/1ª)",
    "TAXA_ATRASO_ETARIO_EF": "Atraso Etário EF (15+ anos)"
}

corr_matrix = df_25[list(cols_corr.keys())].rename(columns=cols_corr).corr(method="spearman")

plt.figure(figsize=(11, 8.5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-0.6,
    vmax=1.0,
    center=0,
    linewidths=0.8,
    cbar_kws={"label": "Correlação de Spearman (ρ)"}
)
plt.title("Matriz de Correlação: Infraestrutura Tecnológica × Professores × Estudantes (2025)", fontsize=13.5, fontweight="bold", pad=14)
plt.tight_layout()
plt.show()
